# ETF Snippets - Quick Analysis Playground

This notebook provides an interactive environment for writing quick analysis snippets.

Use the `Snippet` helper class to iterate through tickers and filter by indicators like RSI, EMA, and Supertrend.

## Import Required Libraries

In [ ]:
import sys
sys.path.insert(0, 'src')

import pandas as pd
import numpy as np
from ETF_screener.snippets import Snippet
from ETF_screener.database import ETFDatabase
from ETF_screener.indicators import add_indicators, calculate_rsi

## Connect to ETF Database

In [ ]:
# Initialize snippet helper
snippet = Snippet()

# Get database stats
db = ETFDatabase()
total_tickers = len(db.get_tickers())
print(f"✓ Connected to database")
print(f"✓ Available tickers: {total_tickers}")
db.close()

## Example 1: Find Overbought ETFs (RSI > 70)

In [ ]:
# Quick method - use built-in filter
overbought = snippet.filter_overbought(rsi_threshold=70)

print(f"\n✓ Found {len(overbought)} ETFs with RSI > 70\n")
print("Ticker     RSI")
print("-" * 20)
for ticker, rsi in overbought:
    print(f"{ticker:<10} {rsi:.1f}")

## Example 2: Manual Iteration - Full Control

In [ ]:
# Manually iterate through all tickers
results = []

for ticker in snippet.iterate_tickers():
    try:
        df = snippet.get_data(ticker)
        
        if df.empty or 'RSI' not in df.columns:
            continue
        
        # Get latest RSI
        latest = df.iloc[-1]
        rsi = latest['RSI']
        price = latest['Close']
        
        # Filter by RSI > 70
        if not pd.isna(rsi) and rsi > 70:
            results.append({
                'Ticker': ticker,
                'Price': f"{price:.2f}",
                'RSI': f"{rsi:.1f}"
            })
    except Exception as e:
        pass

# Display results
if results:
    df_results = pd.DataFrame(results).sort_values('RSI', ascending=False)
    print(f"\n✓ Found {len(results)} ETFs with RSI > 70\n")
    print(df_results.to_string(index=False))
else:
    print("No ETFs found with RSI > 70")

## Example 3: Find Oversold ETFs (RSI < 30)

In [ ]:
# Quick method - find oversold
oversold = snippet.filter_oversold(rsi_threshold=30)

print(f"\n✓ Found {len(oversold)} ETFs with RSI < 30\n")
print("Ticker     RSI")
print("-" * 20)
for ticker, rsi in oversold:
    print(f"{ticker:<10} {rsi:.1f}")

## Example 4: Combine Filters - RSI > 70 AND Price > EMA50

In [ ]:
# Find tickers that are:
# 1. Overbought (RSI > 70)
# 2. Still in uptrend (price > EMA50)

results = []

for ticker in snippet.iterate_tickers():
    try:
        df = snippet.get_data(ticker)
        
        if df.empty or 'RSI' not in df.columns or 'EMA_50' not in df.columns:
            continue
        
        latest = df.iloc[-1]
        rsi = latest['RSI']
        price = latest['Close']
        ema = latest['EMA_50']
        
        # Combine conditions
        if (not pd.isna(rsi) and rsi > 70 and 
            not pd.isna(price) and not pd.isna(ema) and price > ema):
            results.append({
                'Ticker': ticker,
                'Price': f"{price:.2f}",
                'EMA50': f"{ema:.2f}",
                'RSI': f"{rsi:.1f}"
            })
    except Exception:
        pass

if results:
    df_results = pd.DataFrame(results).sort_values('RSI', ascending=False)
    print(f"\n✓ Found {len(results)} ETFs with RSI > 70 AND Price > EMA50\n")
    print(df_results.to_string(index=False))
else:
    print("No matching ETFs found")

## Example 5: Advanced - Get Data for Specific Tickers

In [ ]:
# Analyze specific tickers
tickers_to_check = ['EXS1.DE', 'EUNG.DE', 'EONS.DE']

for ticker in tickers_to_check:
    try:
        df = snippet.get_data(ticker)
        if df.empty:
            print(f"{ticker}: No data found")
            continue
        
        latest = df.iloc[-1]
        print(f"\n{ticker}:")
        print(f"  Price: {latest['Close']:.2f}")
        print(f"  RSI: {latest['RSI']:.1f}")
        print(f"  EMA50: {latest['EMA_50']:.2f}")
        print(f"  Supertrend: {latest['Supertrend']:.2f}")
    except Exception as e:
        print(f"{ticker}: Error - {str(e)}")

## Cleanup

In [ ]:
# Close database connection
snippet.close()
print("✓ Database connection closed")